# Data Mining - Lab 9
## Cube space: Data Cube-ийн тооцоолол ба олон хэмжээст шинжилгээ

**Dataset:** `cs_coffee_mini.csv`, `cs_sales.csv`, `cs_dim_item.csv`, `cs_dim_location.csv`, `cs_iceberg_demo.csv`, `cs_closed_demo.csv`, `cs_monthly_item_city.csv`, `cs_loans.csv`

Энэ лаб нь "Cube space: олон хэмжээст өгөгдлийн шинжилгээ" лекцийн агуулгад үндэслэсэн: cuboid, lattice, cell, concept hierarchy, measure, cube-ийн тооцоолол (materialization, iceberg, BUC, closed cell), OLAP архитектур, cube space дахь mining (prediction cube, discovery-driven exploration).

| Файл | Мөр | Тайлбар |
|---|---|---|
| `cs_coffee_mini` | 8 | Лекцийн кофе шопын мини куб (салбар x улирал x бараа) |
| `cs_sales` | 3000 | 2026 оны 1-6 сарын электроникийн борлуулалтын гүйлгээ |
| `cs_dim_item` | 8 | Барааны dimension (төрөл, брэнд, нийлүүлэгчийн төрөл, нэгж үнэ) |
| `cs_dim_location` | 9 | Байршлын dimension (дүүрэг, хот, улс) |
| `cs_iceberg_demo` | 10 | Лекцийн Iceberg/BUC дасгалын бичлэгүүд |
| `cs_closed_demo` | 5 | Closed cell ба quotient cube-ийн жижиг жишээ |
| `cs_monthly_item_city` | 168 | Бараа x сар x хотын борлуулалт (`month = 0` нь 2025 оны 12-р сар, суурь) |
| `cs_loans` | 2370 | Зээлийн шийдвэрийн өгөгдөл (хот x жил), prediction cube-д |

### Lab зорилго
- Cell (base, aggregate, apex), ancestor ба descendant ойлголтыг кодоор ялгах
- Cuboid-ийн lattice, шатлалтай cube-ийн cuboid-ийн тоог тооцох, `GROUP BY CUBE`-ийг хэрэгжүүлэх
- Concept hierarchy (schema ба set grouping) ашиглан roll-up, drill-down хийх
- Distributive, algebraic, holistic measure-ийн нэгтгэлийн дүрмийг шалгах
- Cuboid сонгох, materialization, Iceberg cube, BUC, closed cell, quotient cube-ийг хэрэгжүүлэх
- ROLAP ба MOLAP-ийг харьцуулах
- Discovery-driven exploration ба prediction cube-ээр cube space дотор mining хийх

Даалгавар бүр **TASK** гэсэн гарчигтай. Код бүрэн өгөгдөөгүй: функцийн нэр, хувьсагчийн нэр, `# TODO` маягийн заавар өгсөн тул `...` болон тайлбарын дагуу кодыг **өөрөө бичнэ**. Хариугаа багшид үзүүлж шалгуулна.

## 0) Өгөгдөл унших

Гүйлгээний хүснэгтийг `cs_dim_item`, `cs_dim_location`-тэй холбож нэг өргөн хүснэгт (`df`) гаргана. Үүн дээр сарын түвшинг (`month`) нэмнэ.

In [57]:
import pandas as pd
import numpy as np
import sqlite3, math
from itertools import combinations

sales    = pd.read_csv("cs_sales.csv")
dim_item = pd.read_csv("cs_dim_item.csv")
dim_loc  = pd.read_csv("cs_dim_location.csv")

df = (sales.merge(dim_item, on="item", validate="many_to_one")
           .merge(dim_loc, on="district", validate="many_to_one"))
df["date"]  = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.strftime("%Y-%m")
print(df.shape, df["sales"].sum())
df.head()

(3000, 16) 2602025


,sale_id,date,item,district,customer_group,units,sales,item_key,type,brand,supplier_type,unit_price,location_key,city,country,month
0,1,2026-01-01,Гар,Чаоян,Хувь хүн,4,200,7,Дагалдах хэрэгсэл,Монгол Тех,Дотоодын,50,7,Бээжин,Хятад,2026-01
1,2,2026-01-01,Монитор,Баянзүрх,Бизнес,2,600,2,Компьютер,Dell,Гадаадын,300,3,Улаанбаатар,Монгол,2026-01
2,3,2026-01-01,Гар,Дархан-Уул,Хувь хүн,1,50,7,Дагалдах хэрэгсэл,Монгол Тех,Дотоодын,50,4,Дархан,Монгол,2026-01
3,4,2026-01-01,Хулгана,Хан-Уул,Хувь хүн,5,125,8,Дагалдах хэрэгсэл,Монгол Тех,Дотоодын,25,2,Улаанбаатар,Монгол,2026-01
4,5,2026-01-01,Утас,Чаоян,Хувь хүн,1,400,4,Гар утас ба таблет,Samsung,Гадаадын,400,7,Бээжин,Хятад,2026-01


## 1) Мини куб: Roll-up, slice, dice (лекцийн дасгал)

Кофе шопын борлуулалт (мянган ₮). Dimension: `branch`, `quarter`, `item`. Measure: `sales`.

### TASK 1: Мини кубийн 6 асуулт

1. **Roll-up**: салбар бүрийн нийт борлуулалт.
2. **Slice**: `quarter = Q2` хүснэгтийг (мөр = салбар, багана = бараа) гарга.
3. **Dice**: `branch = УБ` ба `item = Кофе`. Улирал бүрийн утгыг гарга.
4. **Apex**: `(*, *, *)` cell-ийн утга.
5. **Cuboid**: энэ кубэд хэдэн cuboid байх вэ?
6. **Exception**: Q1-ээс Q2 хүртэл хамгийн их хувиар өссөн (салбар, бараа) cell аль вэ?

In [58]:
coffee = pd.read_csv("cs_coffee_mini.csv")
# 1. Roll-up
print("1. Roll-up (салбар бүрийн нийт борлуулалт)")
print(coffee.groupby("branch")["sales"].sum().to_string())
# 2. Slice: quarter = Q2
print("\n2. Slice: quarter = Q2")
q2 = coffee[coffee["quarter"] == "Q2"].pivot_table(index="branch", columns="item", values="sales", aggfunc="sum")
print(q2.to_string())
# 3. Dice
print("\n3. Dice: УБ x Кофе")
print(coffee[(coffee["branch"] == "УБ") & (coffee["item"] == "Кофе")].set_index("quarter")["sales"].to_string())
# 4. Apex ба cuboid-ийн тоо
n = 3
print("\n4. Apex (*,*,*) =", coffee["sales"].sum(), "| cuboid-ийн тоо = 2**%d =" % n, 2 ** n)
# 5. Exception
pv = coffee.pivot_table(index=["branch", "item"], columns="quarter", values="sales")
pv["growth_%"] = ((pv["Q2"] / pv["Q1"] - 1) * 100).round(2)
print("\n5. Q1 -> Q2 өсөлт (%)")
print(pv.to_string())
print("Хамгийн их өссөн:", pv["growth_%"].idxmax(), "--", pv["growth_%"].max(), "%")

1. Roll-up (салбар бүрийн нийт борлуулалт)
branch
Дархан    200
УБ        400

2. Slice: quarter = Q2
item    Кофе  Цай
branch           
Дархан    65   45
УБ       150   70

3. Dice: УБ x Кофе
quarter
Q1    120
Q2    150

4. Apex (*,*,*) = 600 | cuboid-ийн тоо = 2**3 = 8

5. Q1 -> Q2 өсөлт (%)
quarter         Q1     Q2  growth_%
branch item                        
Дархан Кофе   50.0   65.0     30.00
       Цай    40.0   45.0     12.50
УБ     Кофе  120.0  150.0     25.00
       Цай    60.0   70.0     16.67
Хамгийн их өссөн: ('Дархан', 'Кофе') -- 30.0 %


## 2) Cell: base, aggregate, ancestor, descendant

Cell нь cuboid доторх нэг мөр. `*` нь тухайн dimension-ийг нэгтгэсэн (all) гэсэн тэмдэг.

| Cell | Төрөл |
|---|---|
| `(Jan, Chicago, Business)` | base cell (бүх dimension утгатай) |
| `(Jan, *, Business)` | 2-D aggregate cell |
| `(*, *, *)` | apex cell (0-D) |

**Ancestor** (өвөг) нь илүү ерөнхий, **descendant** (үр удам) нь илүү нарийвчилсан cell. `(Jan, *, *)` нь `(Jan, Chicago, Business)`-ийн ancestor.

### TASK 2: Бүх cell-ийг гаргаж ангилах

1. `cube_cells` функцээр мини кубийн **бүх cell** (base, aggregate, apex)-ийг гарга. Нийт cell-ийн тоо `(n₁+1)(n₂+1)(n₃+1)` томьёотой тэнцэх ёстой.
2. `classify` функцийг бөглөж лекцийн Жишээ 3.9-ийн 5 cell-ийг ангил.
3. `is_ancestor` функцээр `(УБ, Q1, Кофе)` cell-ийн ancestor-уудыг, `(УБ, *, *)` cell-ийн descendant-уудыг ол.

In [59]:
dims = ["branch", "quarter", "item"]
def cube_cells(df, dims, measure):
    parts = []
    for k in range(len(dims), -1, -1):
        for sub in combinations(dims, k):
            if k == 0:                                  
                row = {d: "*" for d in dims}
                row[measure] = df[measure].sum()
                parts.append(pd.DataFrame([row]))
            else:
                g = df.groupby(list(sub), as_index=False)[measure].sum()
                for d in dims:
                    if d not in sub:
                        g[d] = "*"
                parts.append(g[dims + [measure]])
    return pd.concat(parts, ignore_index=True)
cells = cube_cells(coffee, dims, "sales")
n_formula = math.prod(coffee[d].nunique() + 1 for d in dims)
print("1. Нийт cell:", len(cells), "| (n1+1)(n2+1)(n3+1) =", n_formula, "| тэнцэх үү:", len(cells) == n_formula)
def classify(cell):
    k = sum(v != "*" for v in cell)
    if k == len(cell):
        return "base cell"
    if k == 0:
        return "apex cell"
    return f"{k}-D aggregate cell"
ex = [("Jan", "*", "*"), ("*", "Chicago", "*"), ("Jan", "*", "Business"), ("Jan", "Chicago", "Business"), ("*", "*", "*")]
print("\n2. Жишээ 3.9")
for c in ex:
    print(c, "->", classify(c))
def is_ancestor(a, b):
    return a != b and all(x == "*" or x == y for x, y in zip(a, b))
all_cells = [tuple(r) for r in cells[dims].itertuples(index=False)]
target = ("УБ", "Q1", "Кофе")
print("\n3. (УБ, Q1, Кофе)-ийн ancestor-ууд:")
for c in all_cells:
    if is_ancestor(c, target):
        print("  ", c)
target2 = ("УБ", "*", "*")
desc = [c for c in all_cells if is_ancestor(target2, c)]
print("(УБ, *, *)-ийн descendant-ууд (%d):" % len(desc))
for c in desc:
    print("  ", c)

1. Нийт cell: 27 | (n1+1)(n2+1)(n3+1) = 27 | тэнцэх үү: True

2. Жишээ 3.9
('Jan', '*', '*') -> 1-D aggregate cell
('*', 'Chicago', '*') -> 1-D aggregate cell
('Jan', '*', 'Business') -> 2-D aggregate cell
('Jan', 'Chicago', 'Business') -> base cell
('*', '*', '*') -> apex cell

3. (УБ, Q1, Кофе)-ийн ancestor-ууд:
   ('УБ', 'Q1', '*')
   ('УБ', '*', 'Кофе')
   ('*', 'Q1', 'Кофе')
   ('УБ', '*', '*')
   ('*', 'Q1', '*')
   ('*', '*', 'Кофе')
   ('*', '*', '*')
(УБ, *, *)-ийн descendant-ууд (8):
   ('УБ', 'Q1', 'Кофе')
   ('УБ', 'Q1', 'Цай')
   ('УБ', 'Q2', 'Кофе')
   ('УБ', 'Q2', 'Цай')
   ('УБ', 'Q1', '*')
   ('УБ', 'Q2', '*')
   ('УБ', '*', 'Кофе')
   ('УБ', '*', 'Цай')


## 3) Cuboid-ийн lattice ба тоо

Dimension-ы дэд олонлог бүрээр group-by хийвэл нэг cuboid үүснэ. Эдгээрийн lattice нь data cube.

- Hierarchy-гүй n dimension: **2ⁿ** cuboid
- Hierarchy-тай: **T = ∏ (Lᵢ + 1)**, Lᵢ нь i-р dimension-ийн түвшний тоо, `+1` нь `all`

### TASK 3: Lattice ба cuboid-ийн тоо

1. 4 dimension (`time`, `item`, `location`, `supplier`)-ийн lattice-ийг 0-D-ээс 4-D хүртэл түвшин бүрээр тоол.
2. Лекцийн жишээ: `time` (day < month < year, L = 3), `location` (city < country, L = 2), `item` (type, L = 1). Нийт cuboid хэд вэ?
3. 10 dimension, тус бүр 5 түвшинтэй бол хэд вэ?
4. Манай схем: `time` (day, month, year), `location` (district, city, country), `item` (item, type), `customer_group` (1 түвшин). Нийт cuboid хэд вэ?
5. Яагаад бүгдийг урьдчилан тооцоолох (full materialization) практикт боломжгүй вэ? (үгээр)

- Cuboid-ийн тоо dimension болон түвшний тоогоор ХУРДАН (үржвэрээр) өснө: 10 dimension x 5 түвшин = 60 сая гаруй cuboid.
   Cuboid бүр өөрийн мөрүүдтэй тул хадгалалт, тооцоолох хугацаа, шинэчлэлтийн өртөг дэндүү их болно.
   Түүнээс гадна өндөр хэмжээст cube маш сийрэг (sparse) байдаг тул ихэнх cell нь хоосон, үр ашиггүй.

In [60]:
dims4 = ["time", "item", "location", "supplier"]
print("1. Lattice (4 dimension)")
tot = 0
for k in range(len(dims4) + 1):
    cs = list(combinations(dims4, k))
    tot += len(cs)
    print(f"  {k}-D: {len(cs)} cuboid", cs if k in (0, 4) else "")
print("  Нийт =", tot, "= 2**4 =", 2 ** 4)
hier = {"time": 3, "location": 2, "item": 1}
print("\n2. Лекцийн жишээ: T =", " x ".join(str(L + 1) for L in hier.values()), "=", math.prod(L + 1 for L in hier.values()))
print("3. 10 dimension x 5 түвшин: 6**10 =", 6 ** 10, f"({6 ** 10:,})")
ours = {"time": 3, "location": 3, "item": 2, "customer_group": 1}
print("4. Манай схем: T =", " x ".join(str(L + 1) for L in ours.values()), "=", math.prod(L + 1 for L in ours.values()))

1. Lattice (4 dimension)
  0-D: 1 cuboid [()]
  1-D: 4 cuboid 
  2-D: 6 cuboid 
  3-D: 4 cuboid 
  4-D: 1 cuboid [('time', 'item', 'location', 'supplier')]
  Нийт = 16 = 2**4 = 16

2. Лекцийн жишээ: T = 4 x 3 x 2 = 24
3. 10 dimension x 5 түвшин: 6**10 = 60466176 (60,466,176)
4. Манай схем: T = 4 x 4 x 3 x 2 = 96


## 4) GROUP BY CUBE

SQL-ийн `GROUP BY CUBE (month, city)` нь 2² = 4 cuboid-ийг нэг дор гаргана: `(month, city)`, `(month, NULL)`, `(NULL, city)`, `(NULL, NULL)`. SQL дээр `*` (all)-ийн оронд **NULL** гарна.

### TASK 4: CUBE-ийг өөрөө хэрэгжүүлэх

1. `cube_groupby` функц бич: dimension-ы бүх дэд олонлог тус бүрээр бүлэглэж, нэгтгэгдсэн dimension-д `NaN` (NULL) тавь.
2. `df` дээр `["month", "city"]`-ээр CUBE хий. Cuboid тус бүрд хэдэн мөр гарсныг тоол.
3. Apex мөр (бүх dimension NULL) нийт борлуулалттай тэнцэх ёстой. Шалга.
4. SQLite нь `CUBE`-г дэмждэггүй. `UNION ALL`-аар ижил үр дүнг SQL-ээр гаргаж, pandas-ийн үр дүнтэй харьцуул.

In [61]:
def cube_groupby(df, dims, measure, agg="sum"):
    parts = []
    for k in range(len(dims), -1, -1):
        for sub in combinations(dims, k):
            if k == 0:
                row = {d: np.nan for d in dims}
                row[measure] = getattr(df[measure], agg)()
                parts.append(pd.DataFrame([row]))
            else:
                g = df.groupby(list(sub), as_index=False)[measure].agg(agg)
                for d in dims:
                    if d not in sub:
                        g[d] = np.nan
                parts.append(g[dims + [measure]])
    return pd.concat(parts, ignore_index=True)
# 1. CUBE 
cube = cube_groupby(df, ["month", "city"], "sales")
cube["cuboid"] = cube[["month", "city"]].notna().apply(lambda r: "(" + ", ".join(c for c in ["month", "city"] if r[c]) + ")", axis=1)
cube["cuboid"] = cube["cuboid"].replace("()", "apex")
print("1. Cuboid тус бүрийн мөрийн тоо")
print(cube["cuboid"].value_counts().to_string())
print("Нийт мөр:", len(cube))
# 2. Apex 
apex = cube[cube["cuboid"] == "apex"]["sales"].iloc[0]
print("\n2. Apex =", apex, "| нийт борлуулалт =", df["sales"].sum(), "| тэнцэх үү:", apex == df["sales"].sum())
# 3. SQL UNION ALL
con = sqlite3.connect(":memory:")
df[["month", "city", "sales"]].to_sql("sales_table", con, index=False)
q = """
SELECT month, city, SUM(sales) AS sales FROM sales_table GROUP BY month, city
UNION ALL
SELECT month, NULL AS city, SUM(sales) FROM sales_table GROUP BY month
UNION ALL
SELECT NULL AS month, city, SUM(sales) FROM sales_table GROUP BY city
UNION ALL
SELECT NULL AS month, NULL AS city, SUM(sales) FROM sales_table
"""
sql = pd.read_sql(q, con)
a = cube[["month", "city", "sales"]].fillna("*").sort_values(["month", "city"]).reset_index(drop=True)
b = sql.fillna("*").sort_values(["month", "city"]).reset_index(drop=True)
print("\n3. SQL мөрийн тоо:", len(sql), "| pandas мөрийн тоо:", len(a))
print("Ижил үр дүн эсэх:", (a["sales"].values == b["sales"].values).all() and (a[["month","city"]].values == b[["month","city"]].values).all())

1. Cuboid тус бүрийн мөрийн тоо
cuboid
(month, city)    36
(month)           6
(city)            6
apex              1
Нийт мөр: 49

2. Apex = 2602025 | нийт борлуулалт = 2602025 | тэнцэх үү: True

3. SQL мөрийн тоо: 49 | pandas мөрийн тоо: 49
Ижил үр дүн эсэх: True


## 5) Concept hierarchy: roll-up ба drill-down

Concept hierarchy нь доод түвшний ойлголтыг дээд түвшний ойлголтод холбох дараалал. Roll-up бол модоор дээш (дүүрэг -> хот -> улс), drill-down бол доош.

| Төрөл | Тайлбар | Жишээ |
|---|---|---|
| Schema hierarchy | Атрибутын бүрэн эсвэл хэсэгчилсэн дараалал | `day < month < quarter < year`; `day < week` |
| Set grouping hierarchy | Утгыг мужаар бүлэглэнэ | `price`: `($0..$200]`, `($200..$400]` |

Интервалд `(` нь эхлэх утгыг оруулахгүй, `]` нь төгсгөлийн утгыг оруулна.

### TASK 5: Шатлал бүрээр нэгтгэх

1. **Байршил**: борлуулалтыг `country`, дараа нь `city`, дараа нь `district`-ээр гарга. Улс бүрийн нийлбэр хот бүрийнхийн нийлбэртэй тэнцэх ёстой.
2. **Drill-down**: `Монгол` улсаас эхлэж хот руу, хамгийн их хотоос дүүрэг рүү задал.
3. **Хугацаа**: `quarter` (Q1, Q2) баганыг нэм. Улирлаар нэгтгэ.
4. **Хэсэгчилсэн дараалал**: `week` нь `month`-ийн дотор бүрэн багтдаггүй. Нэгээс олон сард хамаарсан ISO долоо хоногийг тоол.
5. **Set grouping**: барааны `unit_price`-ийг 200 нэгжийн интервалаар `pd.cut`-аар бүлэглэ (баруун тал хаалттай). Зөвхөн хэдэн барааны үнэ 200 дээр яг тохиолдож, ямар интервалд орохыг шалга. Дараа нь 100 нэгжийн интервалаар нарийвчил (drill-down).

In [62]:
# 1. Байршил
by_country = df.groupby("country")["sales"].sum()
by_city = df.groupby(["country", "city"])["sales"].sum()
by_district = df.groupby(["country", "city", "district"])["sales"].sum()
print("1. Улс:"); print(by_country.to_string())
print("\nХот:"); print(by_city.to_string())
print("\nДүүрэг:"); print(by_district.to_string())
ok1 = np.allclose(by_country, by_city.groupby("country").sum())
ok2 = np.allclose(by_city, by_district.groupby(["country", "city"]).sum())
print("\nУлс = хотуудын нийлбэр:", ok1, "| Хот = дүүргүүдийн нийлбэр:", ok2)
# 2. Drill-down
print("\n2. Drill-down: Монгол")
mn = df[df["country"] == "Монгол"]
cities = mn.groupby("city")["sales"].sum().sort_values(ascending=False)
print(cities.to_string())
top_city = cities.index[0]
print("\nХамгийн их хот:", top_city)
print(mn[mn["city"] == top_city].groupby("district")["sales"].sum().sort_values(ascending=False).to_string())
# 3. Улирал
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)
print("\n3. Улирлаар:")
print(df.groupby("quarter")["sales"].sum().to_string())
# 4. ISO долоо хоног
iso = df["date"].dt.isocalendar()
df["iso_year_week"] = iso["year"].astype(str) + "-W" + iso["week"].astype(str).str.zfill(2)
wk = df.groupby("iso_year_week")["month"].nunique()
print("\n4. Нийт ISO долоо хоног:", len(wk), "| 2+ сард хамаарсан:", int((wk > 1).sum()))
print(wk[wk > 1].index.tolist())
# 5. Set grouping
print("\n5. Set grouping")
dim_item["band200"] = pd.cut(dim_item["unit_price"], bins=[0, 200, 400, 600, 800], right=True)
print(dim_item[["item", "unit_price", "band200"]].to_string(index=False))
exact = dim_item[dim_item["unit_price"] == 200]
print("\nҮнэ яг 200 байх бараа:", exact["item"].tolist(), "->", exact["band200"].astype(str).tolist(), "(баруун тал хаалттай тул (0,200]-д орно)")
df2 = df.merge(dim_item[["item", "band200"]], on="item", how="left")   # unit_price df-д аль хэдийн байгаа
df2["band100"] = pd.cut(df2["unit_price"], bins=list(range(0, 900, 100)), right=True)
g200 = df2.groupby("band200", observed=True).agg(items=("item", "nunique"), sales=("sales", "sum"))
print("\n200-ийн интервал:"); print(g200.to_string())
g100 = df2.groupby("band100", observed=True).agg(items=("item", "nunique"), sales=("sales", "sum"))
print("\n100-ийн интервал (drill-down):"); print(g100.to_string())

1. Улс:
country
Монгол    1887750
Хятад      714275

Хот:
country  city       
Монгол   Дархан          333200
         Улаанбаатар    1131800
         Ховд            149875
         Эрдэнэт         272875
Хятад    Бээжин          472975
         Шанхай          241300

Дүүрэг:
country  city         district  
Монгол   Дархан       Дархан-Уул    333200
         Улаанбаатар  Баянзүрх      308825
                      Сүхбаатар     429450
                      Хан-Уул       393525
         Ховд         Ховд          149875
         Эрдэнэт      Орхон         272875
Хятад    Бээжин       Хайдиан       224225
                      Чаоян         248750
         Шанхай       Пудун         241300

Улс = хотуудын нийлбэр: True | Хот = дүүргүүдийн нийлбэр: True

2. Drill-down: Монгол
city
Улаанбаатар    1131800
Дархан          333200
Эрдэнэт         272875
Ховд            149875

Хамгийн их хот: Улаанбаатар
district
Сүхбаатар    429450
Хан-Уул      393525
Баянзүрх     308825

3. Улирлаар:
quar

## 6) Measure: distributive, algebraic, holistic

| Төрөл | Жишээ | Хэсгийн үр дүнгээс нийт тооцох |
|---|---|---|
| Distributive | `sum`, `count`, `min`, `max` | Хэсгийн үр дүнг нэгтгэхэд бүтнээр нь бодсонтой адил |
| Algebraic | `avg`, `stddev`, `min_N`, `max_N` | Тогтмол M ширхэг distributive функцээс |
| Holistic | `median`, `mode`, `rank` | Боломжгүй, бүх утгыг хадгална |

### TASK 6: Хэмжүүрийн нэгтгэлийн шалгалт

1. Лекцийн жишээ: Салбар A = 10, 20, 30; Салбар B = 40, 50. `sum`, `count`, `avg`, `median`-ийг A, B, бүгд гэж тус бүр бод. Хэсгээс нэгтгэж болох, болохгүйг тогтоо.
2. Бодит өгөгдөл: барааны `type` бүрийг нэг хэсэг гэж үзэж `sales`-ийн **дундаж**-ыг (a) бүх мөрөөр, (b) `type` бүрийн дундажуудын дундажаар, (c) `type` бүрийн `(sum, count)` хосоор бод. Аль нь зөв бэ? Яагаад (b) буруу байна вэ?
3. **Стандарт хазайлт**-ыг `type` бүрийн `(n, Σx, Σx²)` хосоос бод. Бүтнээр нь бодсонтой тэнцэх ёстой (`ddof = 0`).
4. **median**-ыг `type`-ийн median-уудын median-аар бодоод бүтнийхтэй харьцуул.
5. **max_N** (top 3): `type` бүрийн top 3-ыг нэгтгэж дахин top 3 авбал бүтэн өгөгдлийн top 3-тай тэнцэх үү?

In [63]:
# 1. Лекцийн жишээ
A, B = [10, 20, 30], [40, 50]
ALL = A + B
print("1. Лекцийн жишээ")
rows = []
for name, v in [("A", A), ("B", B), ("Бүгд", ALL)]:
    rows.append([name, sum(v), len(v), np.mean(v), np.median(v)])
print(pd.DataFrame(rows, columns=["хэсэг", "sum", "count", "avg", "median"]).to_string(index=False))
print("sum: A+B =", sum(A) + sum(B), "== бүгд", sum(ALL), "-> distributive")
print("count: A+B =", len(A) + len(B), "== бүгд", len(ALL), "-> distributive")
print("avg: (avg A + avg B)/2 =", (np.mean(A) + np.mean(B)) / 2, "!= бүгд", np.mean(ALL),
      "| (sumA+sumB)/(nA+nB) =", (sum(A) + sum(B)) / (len(A) + len(B)), "-> algebraic")
print("median: (medA+medB)/2 =", (np.median(A) + np.median(B)) / 2, "!= бүгд", np.median(ALL), "-> holistic")
# 2. Бодит өгөгдөл
g = df.groupby("type")["sales"]
part = g.agg(["sum", "count", "mean", "median", "std"])
part["sumsq"] = g.apply(lambda s: (s ** 2).sum())
a = df["sales"].mean()
b = part["mean"].mean()
c = part["sum"].sum() / part["count"].sum()
print("\n2. Дундаж: (a) бүх мөр =", round(a, 4), "| (b) дундажуудын дундаж =", round(b, 4), "| (c) (sum,count) хос =", round(c, 4))
print("   (a)==(c):", np.isclose(a, c), "| (b)==(a):", np.isclose(b, a))
print("   (b) буруу: хэсэг бүрийн мөрийн тоо (жин) өөр өөр, тиймээс жигнэсэн дундаж болох ёстой.")
print(part[["sum", "count", "mean"]].to_string())
# 3. Стандарт хазайлт
n, sx, sxx = part["count"].sum(), part["sum"].sum(), part["sumsq"].sum()
std_parts = math.sqrt(sxx / n - (sx / n) ** 2)
print("\n3. std (ddof=0): хэсгээс =", round(std_parts, 6), "| бүтнээр =", round(df["sales"].std(ddof=0), 6),
      "| тэнцэх үү:", np.isclose(std_parts, df["sales"].std(ddof=0)))
# 4. median
print("\n4. median: бүтнээр =", df["sales"].median(), "| type-ийн median-уудын median =", part["median"].median(),
      "| тэнцэх үү:", df["sales"].median() == part["median"].median())
# 5. max_N (top 3)
top3_parts = pd.concat([s.nlargest(3) for _, s in g]).nlargest(3).tolist()
top3_all = df["sales"].nlargest(3).tolist()
print("\n5. top 3 хэсгээс:", top3_parts, "| бүтнээр:", top3_all, "| тэнцэх үү:", top3_parts == top3_all)

1. Лекцийн жишээ
хэсэг  sum  count  avg  median
    A   60      3 20.0    20.0
    B   90      2 45.0    45.0
 Бүгд  150      5 30.0    30.0
sum: A+B = 150 == бүгд 150 -> distributive
count: A+B = 5 == бүгд 5 -> distributive
avg: (avg A + avg B)/2 = 32.5 != бүгд 30.0 | (sumA+sumB)/(nA+nB) = 30.0 -> algebraic
median: (medA+medB)/2 = 32.5 != бүгд 30.0 -> holistic

2. Дундаж: (a) бүх мөр = 867.3417 | (b) дундажуудын дундаж = 921.3431 | (c) (sum,count) хос = 867.3417
   (a)==(c): True | (b)==(a): False
   (b) буруу: хэсэг бүрийн мөрийн тоо (жин) өөр өөр, тиймээс жигнэсэн дундаж болох ёстой.
                        sum  count         mean
type                                           
Гар утас ба таблет  1172400    879  1333.788396
Дагалдах хэрэгсэл    221125   1144   193.291084
Компьютер           1208500    977  1236.949846

3. std (ddof=0): хэсгээс = 858.234316 | бүтнээр = 858.234316 | тэнцэх үү: True

4. median: бүтнээр = 600.0 | type-ийн median-уудын median = 900.0 | тэнцэх үү: False


## 7) Query-д cuboid сонгох

Query ирэхэд хамгийн бага хэмжээтэй, асуултыг хариулж чадах materialized cuboid-ийг сонгоно.

1. Query-ийн GROUP BY ба WHERE-ээс шаардлагатай түвшнийг тогтооно.
2. Hierarchy-гээр тухайн түвшин эсвэл **илүү нарийвчилсан** түвшний cuboid-ийг хайна (хойно roll-up хийж болно).
3. Хамгийн бага өртөгтэй (мөрийн тоо хамгийн цөөн) cuboid-ийг сонгоно.

Cuboid-ийг `(time түвшин, location түвшин, item түвшин)` гэж тэмдэглэнэ, `None` нь `all`. Жишээ нь `("month", "city", None)`.

### TASK 7: Cuboid сонгогч

Дараах cuboid-ууд materialized: `("day","district","item")`, `("month","city","item")`, `("month","city",None)`, `("month","country","item")`, `("year","country",None)`.

1. `can_answer`: materialized cuboid нь query-г хариулж чадах нөхцөл. Dimension бүрт materialized түвшин query-ийн түвшингээс **нарийн эсвэл тэнцүү** байх ёстой.
2. `size`: cuboid-ийн мөрийн тоо (өгөгдлөөс).
3. Дараах query-үүдэд хариулж чадах cuboid-уудыг жагсааж, хамгийн бага хэмжээтэйг сонго:
   - Q1: `("month", "country", None)`
   - Q2: `("year", "city", "type")`
   - Q3: `(None, None, None)` (apex)
4. Q2-ыг `("month","city",None)` cuboid яагаад хариулж чадахгүй вэ? (үгээр)

-Q2 = ("year","city","type"). ("month","city",None) cuboid-д item dimension 'all' болтлоо нэгтгэгдсэн тул
   барааны мэдээлэл (item/type) алдагдсан. Нэгтгэсэн cuboid-ээс илүү нарийвчилсан түвшин рүү drill-down хийх боломжгүй,
   тиймээс type-аар бүлэглэх асуултад хариулж чадахгүй.

In [ ]:
df["day"]  = df["date"].dt.strftime("%Y-%m-%d")
df["year"] = df["date"].dt.year
LEVELS = {"time": ["day", "month", "year"],
          "location": ["district", "city", "country"],
          "item": ["item", "type"]}
DIMS = ["time", "location", "item"]
def rank(dim, level):
    return LEVELS[dim].index(level) if level else len(LEVELS[dim])    # None = all
def can_answer(mat, q):
    return all(rank(d, m) <= rank(d, x) for d, m, x in zip(DIMS, mat, q))
def size(c):
    cols = [x for x in c if x is not None]
    return df.groupby(cols).ngroups if cols else 1
materialized = [("day", "district", "item"), ("month", "city", "item"), ("month", "city", None),
                ("month", "country", "item"), ("year", "country", None)]
queries = [("month", "country", None), ("year", "city", "type"), (None, None, None)]
print("Materialized cuboid-ийн хэмжээ:")
for m in materialized:
    print("  ", m, "->", size(m))
for q in queries:
    ok = [m for m in materialized if can_answer(m, q)]
    print("\nQuery", q)
    for m in ok:
        print("   хариулж чадна:", m, "хэмжээ", size(m))
    best = min(ok, key=size)
    print("   => СОНГОСОН:", best, "(", size(best), "мөр )")

Materialized cuboid-ийн хэмжээ:
   ('day', 'district', 'item') -> 2616
   ('month', 'city', 'item') -> 288
   ('month', 'city', None) -> 36
   ('month', 'country', 'item') -> 96
   ('year', 'country', None) -> 2

Query ('month', 'country', None)
   хариулж чадна: ('day', 'district', 'item') хэмжээ 2616
   хариулж чадна: ('month', 'city', 'item') хэмжээ 288
   хариулж чадна: ('month', 'city', None) хэмжээ 36
   хариулж чадна: ('month', 'country', 'item') хэмжээ 96
   => СОНГОСОН: ('month', 'city', None) ( 36 мөр )

Query ('year', 'city', 'type')
   хариулж чадна: ('day', 'district', 'item') хэмжээ 2616
   хариулж чадна: ('month', 'city', 'item') хэмжээ 288
   => СОНГОСОН: ('month', 'city', 'item') ( 288 мөр )

Query (None, None, None)
   хариулж чадна: ('day', 'district', 'item') хэмжээ 2616
   хариулж чадна: ('month', 'city', 'item') хэмжээ 288
   хариулж чадна: ('month', 'city', None) хэмжээ 36
   хариулж чадна: ('month', 'country', 'item') хэмжээ 96
   хариулж чадна: ('year', 'countr

## 8) Materialization: Full, No, Partial

| Арга | Тодорхойлолт | Онцлог |
|---|---|---|
| Full | Бүх cuboid-ийг урьдчилан тооцно | Query хурдан, хадгалалт их |
| No | Нэгтгэлийг хэрэгтэй үед тооцно | Хадгалалт бага, query удаан |
| Partial | Түгээмэл cuboid-уудыг хадгална | Хурд ба хадгалалтын тэнцвэр |

### TASK 8: 3 dimension-ийн cube (item, city, month)

1. 8 cuboid-ийн мөрийн тоог гарга. **Full** materialization-ий нийт хадгалалт, **No** materialization-ий хадгалалт (зөвхөн base cuboid) хэд вэ?
2. Query бүр өөрийн cuboid-ийг хариулахад, хариулж чадах хамгийн жижиг materialized cuboid-ийг уншина гэж үзвэл 8 cuboid бүрийн query-ийн **дундаж өртөг** хэд вэ? (Full ба No-д тус тусад нь.)
3. **Partial**: base-ээс гадна 2 cuboid сонгож (нийт 3), дундаж өртгийг хамгийн бага болгох жагсаалтыг greedy аргаар ол.
4. Partial-ийн хадгалалт ба дундаж өртгийг Full, No-той харьцуул. Лекцийн "зөвхөн 3 cuboid хадгалбал ямар materialization вэ?" асуултад хариул.

In [65]:
D3 = ["item", "city", "month"]
cubs = [frozenset(s) for k in range(len(D3) + 1) for s in combinations(D3, k)]
def sz(c):
    return df.groupby(list(c)).ngroups if c else 1
sizes = {c: sz(c) for c in cubs}
def nm(c):
    return "(" + ", ".join(d for d in D3 if d in c) + ")" if c else "apex"
print("1. 8 cuboid-ийн хэмжээ")
for c in cubs:
    print(f"   {nm(c):25s} {sizes[c]}")
base = frozenset(D3)
print("Full хадгалалт =", sum(sizes.values()), "| No (зөвхөн base) =", sizes[base])
def cost(q, mat):
    return min(sizes[m] for m in mat if q <= m)
def avg_cost(mat):
    return np.mean([cost(q, mat) for q in cubs])
full_avg = avg_cost(cubs)
no_avg = avg_cost([base])
print("\n2. Дундаж өртөг: Full =", round(full_avg, 2), "| No =", round(no_avg, 2))
mat = [base]
print("\n3. Greedy")
for step in range(2):
    cand = [c for c in cubs if c not in mat]
    best = min(cand, key=lambda c: avg_cost(mat + [c]))
    before = avg_cost(mat)
    mat.append(best)
    print(f"   Алхам {step + 1}: {nm(best)} нэмэв | дундаж өртөг {before:.2f} -> {avg_cost(mat):.2f}")
part_store = sum(sizes[m] for m in mat)
part_avg = avg_cost(mat)

print("\n4. Харьцуулалт")
print(pd.DataFrame({
    "арга": ["Full (8 cuboid)", "No (1 cuboid)", "Partial (3 cuboid)"],
    "хадгалалт (мөр)": [sum(sizes.values()), sizes[base], part_store],
    "дундаж өртөг": [round(full_avg, 2), round(no_avg, 2), round(part_avg, 2)]}).to_string(index=False))
print("Сонгосон 3 cuboid:", [nm(m) for m in mat])
print("=> Зөвхөн 3 cuboid хадгалбал энэ нь PARTIAL materialization.")

1. 8 cuboid-ийн хэмжээ
   apex                      1
   (item)                    8
   (city)                    6
   (month)                   6
   (item, city)              48
   (item, month)             48
   (city, month)             36
   (item, city, month)       288
Full хадгалалт = 441 | No (зөвхөн base) = 288

2. Дундаж өртөг: Full = 55.12 | No = 288.0

3. Greedy
   Алхам 1: (city, month) нэмэв | дундаж өртөг 288.00 -> 162.00
   Алхам 2: (item, city) нэмэв | дундаж өртөг 162.00 -> 102.00

4. Харьцуулалт
              арга  хадгалалт (мөр)  дундаж өртөг
   Full (8 cuboid)              441         55.12
     No (1 cuboid)              288        288.00
Partial (3 cuboid)              372        102.00
Сонгосон 3 cuboid: ['(item, city, month)', '(city, month)', '(item, city)']
=> Зөвхөн 3 cuboid хадгалбал энэ нь PARTIAL materialization.


## 9) Iceberg cube, antimonotonicity ба BUC

**Iceberg cube**: `COUNT(*) >= min_sup` нөхцөл хангасан cell-үүдийг л хадгалсан cube.

```
compute cube sales_iceberg as
select month, city, customer_group, count(*)
from salesInfo cube by month, city, customer_group
having count(*) >= min_sup
```

**Antimonotonicity**: нэг cell босго хангахгүй бол түүнээс нарийвчилсан бүх descendant ч хангахгүй. **BUC** (Bottom-Up Computation) нь ингэж **pruning** хийж iceberg cube-ийг үр ашигтай тооцдог.

### TASK 9: Лекцийн дасгал (min_sup = 3) ба BUC

`cs_iceberg_demo.csv` нь 10 бичлэгтэй (item, city): Ном УБ x4, Ном Дархан x1, Дэвтэр УБ x3, Дэвтэр Дархан x2.

1. `(item, city)`-ийн COUNT-ийг гарга. `min_sup = 3` үед Iceberg cube-д аль cell-үүд үлдэх вэ? (CUBE-ийн бүх cuboid-ийг оруул.)
2. `buc` функцийг бөглөж ажиллуул. Аль cell-үүдийн цаашдын салбарыг **prune** хийсэн бэ?
3. Том өгөгдөл: `["month", "city", "customer_group", "item"]`-ээр `min_sup = 30` үед BUC-ийн үр дүн CUBE-ээр бүгдийг тооцоод шүүсэнтэй ижил эсэхийг шалга. Хэдэн cell үлдэв, хэдэн partition хянагдав, нийт cell хэд байв?
4. Antimonotonicity-г шалга: iceberg cell бүрийн ancestor бүр мөн iceberg-д орсон эсэх.

In [66]:
demo = pd.read_csv("cs_iceberg_demo.csv")
min_sup = 3
demo["cnt"] = 1
cd_all = cube_cells(demo, ["item", "city"], "cnt")
print("1. Бүх cell ба COUNT"); print(cd_all.to_string(index=False))
ice = cd_all[cd_all["cnt"] >= min_sup]
print("\nIceberg cube (COUNT >= %d):" % min_sup); print(ice.to_string(index=False))
def buc(df, dims, min_sup):
    result, pruned, counter = [], [], {"parts": 0}
    def rec(part, prefix, start):
        cell = tuple(prefix.get(d, "*") for d in dims)
        result.append((cell, len(part)))
        for j in range(start, len(dims)):
            d = dims[j]
            for val, sub in part.groupby(d):
                counter["parts"] += 1
                new = dict(prefix); new[d] = val
                c = tuple(new.get(x, "*") for x in dims)
                if len(sub) >= min_sup:
                    rec(sub, new, j + 1)
                else:
                    pruned.append((c, len(sub)))      # prune
    if len(df) >= min_sup:
        rec(df, {}, 0)
    return result, pruned, counter["parts"]
# 2.
res, pr, parts = buc(demo, ["item", "city"], 3)
print("\n2. BUC үр дүн:")
for c, n_ in res: print("   ", c, n_)
print("Prune хийсэн cell:")
for c, n_ in pr: print("   ", c, "count =", n_)
print("Хянасан partition:", parts)
print("BUC == CUBE-ээс шүүсэн:", {c for c, _ in res} == {tuple(r) for r in ice[["item", "city"]].itertuples(index=False)})
# 3. Том өгөгдөл
d4 = ["month", "city", "customer_group", "item"]
ms = 30
res4, pr4, parts4 = buc(df, d4, ms)
df["cnt"] = 1
full = cube_cells(df, d4, "cnt")
ice4 = full[full["cnt"] >= ms]
s_buc = {(c, n_) for c, n_ in res4}
s_cube = {(tuple(r[:4]), r[4]) for r in ice4.itertuples(index=False)}
print("\n3. min_sup = 30, 4 dimension")
print("BUC == CUBE шүүлт:", s_buc == s_cube)
print("Iceberg cell:", len(res4), "| хянасан partition:", parts4, "| нийт cell (CUBE):", len(full))
print(f"Iceberg нь нийт cell-ийн {len(res4) / len(full) * 100:.1f}%")
# 4. Antimonotonicity
ice_set = {c for c, _ in res4}
viol = 0
for c in ice_set:
    nz = [i for i, v in enumerate(c) if v != "*"]
    for k in range(len(nz)):
        for sub in combinations(nz, k):
            anc = tuple(c[i] if i in sub else "*" for i in range(len(c)))
            if anc not in ice_set:
                viol += 1
print("\n4. Antimonotonicity зөрчил:", viol)

1. Бүх cell ба COUNT
  item   city  cnt
Дэвтэр Дархан    2
Дэвтэр     УБ    3
   Ном Дархан    1
   Ном     УБ    4
Дэвтэр      *    5
   Ном      *    5
     * Дархан    3
     *     УБ    7
     *      *   10

Iceberg cube (COUNT >= 3):
  item   city  cnt
Дэвтэр     УБ    3
   Ном     УБ    4
Дэвтэр      *    5
   Ном      *    5
     * Дархан    3
     *     УБ    7
     *      *   10

2. BUC үр дүн:
    ('*', '*') 10
    ('Дэвтэр', '*') 5
    ('Дэвтэр', 'УБ') 3
    ('Ном', '*') 5
    ('Ном', 'УБ') 4
    ('*', 'Дархан') 3
    ('*', 'УБ') 7
Prune хийсэн cell:
    ('Дэвтэр', 'Дархан') count = 2
    ('Ном', 'Дархан') count = 1
Хянасан partition: 8
BUC == CUBE-ээс шүүсэн: True

3. min_sup = 30, 4 dimension
BUC == CUBE шүүлт: True
Iceberg cell: 310 | хянасан partition: 1135 | нийт cell (CUBE): 1636
Iceberg нь нийт cell-ийн 18.9%

4. Antimonotonicity зөрчил: 0


## 10) Closed cell, quotient cube, shell fragment

- **Closed cell**: өөрөөсөө илүү нарийвчилсан, **яг ижил суурь нүднүүдийг (base cells) хамарсан** өөр cell байхгүй cell.
- **Quotient cube**: зөвхөн closed cell-үүдээс бүрдсэн шахсан cube.
- **Shell fragment**: dimension-уудыг жижиг бүлэгт хувааж зөвхөн тэдгээрийн cuboid-ийг урьдчилан тооцох арга.

### TASK 10: Closed cell олох

`cs_closed_demo.csv` нь 5 суурь мөртэй: Дархан Q1 Кофе, Дархан Q1 Цай, УБ Q2 Кофе, УБ Q2 Цай, УБ Q3 Цай.

1. `cube_cells`-ээр (TASK 2) бүх non-empty cell-ийг гарга. Нийт хэдэн cell байна вэ?
2. Cell бүрийн **cover** (хамарсан суурь мөрийн индексийн олонлог)-ийг тооц.
3. Closed cell нь: өөрийн cover-тэй **яг ижил cover-тэй descendant байхгүй** cell. Closed cell-үүдийг ол.
4. `(Дархан, *, *)` closed уу? Яагаад? Түүнийг төлөөлөх closed cell аль вэ?
5. Quotient cube-д хэдэн cell үлдэх вэ? Шахалтын харьцаа хэд вэ?
6. Shell fragment: 4 dimension-ийг `{time, item}` ба `{location, supplier}` хоёр бүлэгт хувааж, бүлэг доторх cuboid (apex-гүй)-ийг л тооцвол хэдэн cuboid тооцох вэ? Бүтэн cube-тэй (apex-гүй 15) харьцуул.

In [67]:
cd = pd.read_csv("cs_closed_demo.csv")
dims_c = ["branch", "quarter", "item"]
cells_c = cube_cells(cd, dims_c, "sales")
print("1. Non-empty cell:", len(cells_c))
all_c = [tuple(r) for r in cells_c[dims_c].itertuples(index=False)]
print("   (бүтэн cross: (2+1)(3+1)(2+1) =", 3 * 4 * 3, "боловч хоосон cell-үүд хасагдсан)")
def covers(cell):
    mask = np.ones(len(cd), dtype=bool)
    for d, v in zip(dims_c, cell):
        if v != "*":
            mask &= (cd[d] == v).values
    return frozenset(np.where(mask)[0].tolist())
def is_closed(cell):
    cv = covers(cell)
    return not any(is_ancestor(cell, d) and covers(d) == cv for d in all_c)
closed = [c for c in all_c if is_closed(c)]
print("\n3. Closed cell-үүд (%d):" % len(closed))
for c in closed:
    print("  ", c, "cover =", sorted(covers(c)), "sales =", int(cells_c[(cells_c[dims_c] == list(c)).all(axis=1)]["sales"].iloc[0]))
c0 = ("Дархан", "*", "*")
same = [d for d in all_c if is_ancestor(c0, d) and covers(d) == covers(c0)]
print("\n4.", c0, "closed уу:", is_closed(c0))
print("   Ижил cover-тэй descendant:", same)
rep = [c for c in closed if covers(c) == covers(c0)]
print("   Түүнийг төлөөлөх closed cell:", rep)
print("\n5. Quotient cube cell:", len(closed), "/", len(all_c), "| шахалтын харьцаа =", round(len(all_c) / len(closed), 2), "x (",
      f"{(1 - len(closed) / len(all_c)) * 100:.1f}% хэмнэлт)")
frag = 0
for grp in (["time", "item"], ["location", "supplier"]):
    frag += sum(1 for k in range(1, len(grp) + 1) for _ in combinations(grp, k))
print("\n6. Shell fragment cuboid:", frag, "| бүтэн cube (apex-гүй):", 2 ** 4 - 1, "| хэмнэлт:", 2 ** 4 - 1 - frag)

1. Non-empty cell: 25
   (бүтэн cross: (2+1)(3+1)(2+1) = 36 боловч хоосон cell-үүд хасагдсан)

3. Closed cell-үүд (12):
   ('Дархан', 'Q1', 'Кофе') cover = [0] sales = 50
   ('Дархан', 'Q1', 'Цай') cover = [1] sales = 40
   ('УБ', 'Q2', 'Кофе') cover = [2] sales = 150
   ('УБ', 'Q2', 'Цай') cover = [3] sales = 70
   ('УБ', 'Q3', 'Цай') cover = [4] sales = 30
   ('Дархан', 'Q1', '*') cover = [0, 1] sales = 90
   ('УБ', 'Q2', '*') cover = [2, 3] sales = 220
   ('УБ', '*', 'Цай') cover = [3, 4] sales = 100
   ('УБ', '*', '*') cover = [2, 3, 4] sales = 250
   ('*', '*', 'Кофе') cover = [0, 2] sales = 200
   ('*', '*', 'Цай') cover = [1, 3, 4] sales = 140
   ('*', '*', '*') cover = [0, 1, 2, 3, 4] sales = 340

4. ('Дархан', '*', '*') closed уу: False
   Ижил cover-тэй descendant: [('Дархан', 'Q1', '*')]
   Түүнийг төлөөлөх closed cell: [('Дархан', 'Q1', '*')]

5. Quotient cube cell: 12 / 25 | шахалтын харьцаа = 2.08 x ( 52.0% хэмнэлт)

6. Shell fragment cuboid: 6 | бүтэн cube (apex-гүй): 15

## 11) OLAP серверийн архитектур: ROLAP ба MOLAP

| Архитектур | Хадгалалт | Нэгтгэл |
|---|---|---|
| ROLAP | Relational хүснэгт | SQL |
| MOLAP | Олон хэмжээст массив | Массивын индекс, тэнхлэгээр нэгтгэх |
| HOLAP | Хослуулсан | Нэгтгэсэн хэсэг MOLAP, дэлгэрэнгүй хэсэг ROLAP |

### TASK 11: Ижил асуулга, хоёр архитектур

1. **ROLAP**: SQLite-д `fact` хүснэгт үүсгэж `GROUP BY city, month`-ээр SUM гарга.
2. **MOLAP**: `item x city x month` dense numpy массив үүсгэ (`np.add.at`). `item` тэнхлэгийг roll-up хий (`sum(axis=0)`).
3. Хоёр үр дүн ижил эсэхийг шалга.
4. **Sparsity**: `district x item x day x customer_group` dimension-ийн dense массивын нийт cell-ийн тоо, **хоосон биш** cell-ийн тоо, хоосон биш cell-ийн хувийг гарга. Яагаад олон dimension-тэй cube-д MOLAP-ийн dense массив хэмжээ зарцуулагддаг вэ? HOLAP яагаад үүнийг засдаг вэ? (үгээр)

--MOLAP dense массив dimension нэмэх тусам cell-ийн тоо үржвэрээр тэсрэн өсдөг, харин бодит өгөгдөл маш цөөн cell-д л утгатай
   тул ихэнх зай хоосон (0) утгаар дүүрнэ. HOLAP нэгтгэсэн (бага, нягт) хэсгийг MOLAP массивт, сийрэг дэлгэрэнгүй хэсгийг
   ROLAP хүснэгтэд хадгалснаар хоёуланг нь давуу талыг авч, хоосон зайн үрэлгэнийг арилгана.

In [68]:
# 1. ROLAP
con = sqlite3.connect(":memory:")
df[["item", "city", "month", "district", "customer_group", "day", "sales"]].to_sql("fact", con, index=False)
rolap = pd.read_sql("SELECT city, month, SUM(sales) AS sales FROM fact GROUP BY city, month", con)
rolap_p = rolap.pivot(index="city", columns="month", values="sales").fillna(0)
print("1. ROLAP (SQL GROUP BY city, month):", rolap.shape)
# 2. MOLAP
items = sorted(df["item"].unique()); cities = sorted(df["city"].unique()); months = sorted(df["month"].unique())
ii = df["item"].map({v: i for i, v in enumerate(items)}).values
ci = df["city"].map({v: i for i, v in enumerate(cities)}).values
mi = df["month"].map({v: i for i, v in enumerate(months)}).values
cube_arr = np.zeros((len(items), len(cities), len(months)))
np.add.at(cube_arr, (ii, ci, mi), df["sales"].values)
rolled = cube_arr.sum(axis=0)            # item тэнхлэгийг roll-up
print("2. MOLAP массивын хэлбэр:", cube_arr.shape, "-> roll-up item:", rolled.shape)
# 3.
molap_p = pd.DataFrame(rolled, index=cities, columns=months)
print("3. ROLAP == MOLAP:", np.allclose(rolap_p.loc[cities, months].values, molap_p.values))
# 4. Sparsity
d = ["district", "item", "day", "customer_group"]
dense = math.prod(df[x].nunique() for x in d)
nonempty = df.groupby(d).ngroups
print("\n4. Sparsity:", {x: df[x].nunique() for x in d})
print(f"   Dense cell = {dense:,} | хоосон биш = {nonempty:,} | {nonempty / dense * 100:.2f}% (хоосон {100 - nonempty / dense * 100:.2f}%)")

1. ROLAP (SQL GROUP BY city, month): (36, 3)
2. MOLAP массивын хэлбэр: (8, 6, 6) -> roll-up item: (6, 6)
3. ROLAP == MOLAP: True

4. Sparsity: {'district': 9, 'item': 8, 'day': 181, 'customer_group': 3}
   Dense cell = 39,096 | хоосон биш = 2,829 | 7.24% (хоосон 92.76%)


## 12) Discovery-driven exploration

Сая сая cell-ээс сонирхолтойг гараар хайх боломжгүй. Хүлээгдэж буй утгаас ихээр зөрсөн cell-ийг стандартчилсан хазайлтаар (жишээ нь |z| > 2.5) автоматаар тэмдэглэнэ.

- **SelfExp**: cell өөрөө хэр онцгой
- **InExp**: cell-ийн доор (drill-down) онцгой утга нуугдсан эсэх
- **PathExp**: drill-down зам бүрийн онцгой байдал

### TASK 12: SelfExp ба InExp

`cs_monthly_item_city.csv`-д 4 бараа x 6 хот x сар (0 = 2025 оны 12-р сар).

1. Барааны сарын борлуулалтын өөрчлөлтийг (%) гарга: `chg` (бараа x сар 1-6). Лекцийн хүснэгттэй адил эсэхийг шалга.
2. **SelfExp**: хүлээгдэх утгыг мөрийн дундаж + баганын дундаж - нийт дундаж гэж үзэж, үлдэгдэл (residual)-ийг стандартчил. Хамгийн онцгой 4 cell, мөн |z| > 2 болох cell-ийн тоог ол (жижиг хүснэгт тул босгыг 2 гэж авна).
3. **InExp**: cell бүрийг хот руу drill-down хийж, хотын өөрчлөлт нь улсын өөрчлөлтөөс (`chg`) хэр зөрөхийг стандартчил. Бараа x сар бүрт хотуудын хамгийн их |z|-ийг ол.
4. SelfExp өндөр боловч InExp нам, мөн SelfExp нам боловч InExp өндөр cell байна уу? Тэдгээрийг тайлбарла. (Лекцийн Принтер 4-р сар жишээ)

In [69]:
mo = pd.read_csv("cs_monthly_item_city.csv")
# 1.
nat = mo.groupby(["item", "month"])["sales"].sum().unstack("month")
chg = (nat.pct_change(axis=1) * 100).iloc[:, 1:]
print("1. Улсын нийлбэр:"); print(nat.to_string())
print("\nСарын өөрчлөлт, % (chg):"); print(chg.round(2).to_string())
# 2. SelfExp
exp = chg.mean(axis=1).values[:, None] + chg.mean(axis=0).values[None, :] - chg.values.mean()
resid = chg - exp
z = resid / resid.values.std(ddof=1)
zs = z.stack().rename("z").reset_index()
zs["|z|"] = zs["z"].abs()
print("\n2. SelfExp: хамгийн онцгой 4 cell")
print(zs.sort_values("|z|", ascending=False).head(4).round(2).to_string(index=False))
print("|z| > 2 cell-ийн тоо:", int((z.abs() > 2).sum().sum()), "/", z.size)
# 3. InExp
cm = mo.pivot_table(index=["item", "city"], columns="month", values="sales", aggfunc="sum")
cchg = (cm.pct_change(axis=1) * 100).iloc[:, 1:]
diff = cchg.sub(chg.reindex(cchg.index.get_level_values("item")).values)   # хот - улс
zc = diff / diff.values.std(ddof=1)
zc_long = zc.stack().rename("z").reset_index()
zc_long["|z|"] = zc_long["z"].abs()
idx = zc_long.groupby(["item", "month"])["|z|"].idxmax()
inexp = zc_long.loc[idx].rename(columns={"|z|": "InExp"})
print("\n3. InExp: бараа x сар бүрт хотуудын хамгийн их |z|")
print(inexp.sort_values("InExp", ascending=False).head(8).round(2).to_string(index=False))
top = inexp.sort_values("InExp", ascending=False).iloc[0]
print("Хамгийн онцгой cell:", top["item"], "сар", top["month"], "хот", top["city"], "z =", round(top["z"], 2))
# 4. SelfExp vs InExp
m = zs.merge(inexp[["item", "month", "city", "InExp"]], on=["item", "month"])
m["SelfExp"] = m["|z|"]
print("\n4. SelfExp өндөр (|z|>2) & InExp нам (<1):")
print(m[(m["SelfExp"] > 2) & (m["InExp"] < 1)][["item", "month", "SelfExp", "InExp", "city"]].round(2).to_string(index=False))
print("SelfExp нам (<1) & InExp өндөр (>2):")
print(m[(m["SelfExp"] < 1) & (m["InExp"] > 2)][["item", "month", "SelfExp", "InExp", "city"]].round(2).to_string(index=False))
print("\nЖишээ: Принтер сар 4 -> улсын өөрчлөлт %.2f%%, SelfExp = %.2f, InExp = %.2f (хот: %s)" % (
    chg.loc["Принтер", 4], m[(m["item"] == "Принтер") & (m["month"] == 4)]["SelfExp"].iloc[0],
    m[(m["item"] == "Принтер") & (m["month"] == 4)]["InExp"].iloc[0], m[(m["item"] == "Принтер") & (m["month"] == 4)]["city"].iloc[0]))
print(cchg.loc["Принтер", 4].round(2).to_string())

1. Улсын нийлбэр:
month       0     1     2     3     4     5     6
item                                             
Ноутбук  8018  8177  8422  8506  8846  9024  9295
Принтер  1518  1503  1504  1518  1488  1489  1502
Утас     5919  5979  6098  5243  5350  5403  5564
Чихэвч   1983  2041  2085  2167  2188  2669  2723

Сарын өөрчлөлт, % (chg):
month       1     2      3     4      5     6
item                                         
Ноутбук  1.98  3.00   1.00  4.00   2.01  3.00
Принтер -0.99  0.07   0.93 -1.98   0.07  0.87
Утас     1.01  1.99 -14.02  2.04   0.99  2.98
Чихэвч   2.92  2.16   3.93  0.97  21.98  2.02

2. SelfExp: хамгийн онцгой 4 cell
   item  month     z  |z|
 Чихэвч      5  2.82 2.82
   Утас      3 -2.23 2.23
Ноутбук      5 -1.18 1.18
Принтер      3  1.17 1.17
|z| > 2 cell-ийн тоо: 2 / 24

3. InExp: бараа x сар бүрт хотуудын хамгийн их |z|
   item   city  month      z  InExp
Принтер   Ховд      4 -10.56  10.56
Принтер Бээжин      2  -1.25   1.25
 Чихэвч Бээжин      2   1.

## 13) Prediction cube

Cell бүрт тухайн cell-ийн өгөгдлөөр сургасан загварын **accuracy**-г measure болгон хадгална. Хоёр хэлбэр: (1) загварын accuracy, (2) attribute-ийн predictiveness, өөрөөр хэлбэл attribute-тэй ба гүй загварын accuracy-ийн зөрүү.

`cs_loans.csv`: зээлийн шийдвэр (`approved`). Dimension: `city`, `year`. Feature: `age`, `income`, `debt_ratio`, `gender`.

### TASK 13: Зээлийн prediction cube

1. (`city`, `year`) cell бүр дээр `age`, `income`, `debt_ratio`-оор logistic regression-ийг 5-fold cross validation-ээр сургаж accuracy-г гарга (`StandardScaler` ашигла).
2. Cell бүрт `gender`-ийг нэмж accuracy-г дахин гарга. Зөрүү (`gender_gain`) нь gender-ийн predictiveness.
3. Accuracy-ийн (хот x жил) хүснэгтийг гарга. Загвар хамгийн сул хот аль вэ? Яагаад (cell дэх өгөгдлийн хэмжээ, шуугианыг бод)?
4. Аль хотод `gender`-ийг нэмэхэд accuracy тогтвортой өсч байна вэ? Ховд-д зөрүү яагаад тогтворгүй байна вэ? Энэ нь fairness-ийн ямар сэрэмжлүүлэг вэ? (үгээр)

--gender нэмэхэд accuracy тогтвортой өссөн хот нь Эрдэнэт (+0.035, +0.070, +0.075). Ховдод зөрүү тогтворгүй, сөрөг (-0.056, -0.100): өгөгдөл цөөн тул overfitting болон шуугианаас үүссэн байж болно.
Fairness сэрэмжлүүлэг: Эрдэнэтэд хүйс зээлийн шийдвэрийг таамаглахад тусалж байгаа нь загвар хүйсийг шийдвэрийн хүчин зүйл болгон ашиглаж, ялгаварлан гадуурхах эрсдэлтэй гэсэн үг. Prediction cube нь ийм хот, жилийг илрүүлж өгдөг.

In [70]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
loans = pd.read_csv("cs_loans.csv")
def cell_acc(g, feats):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(model, g[feats], g["approved"], cv=cv, scoring="accuracy").mean()
base_f = ["age", "income", "debt_ratio"]
rows = []
for (city, year), g in loans.groupby(["city", "year"]):
    a0 = cell_acc(g, base_f)                    # 1
    a1 = cell_acc(g, base_f + ["gender"])       # 2
    rows.append({"city": city, "year": year, "n": len(g), "acc": a0, "acc_gender": a1, "gender_gain": a1 - a0})
pc = pd.DataFrame(rows)
print("Accuracy (хот x жил):")
print(pc.pivot(index="city", columns="year", values="acc").round(4).to_string())
print("\nAccuracy gender-тэй:")
print(pc.pivot(index="city", columns="year", values="acc_gender").round(4).to_string())
print("\ngender_gain (accuracy-ийн зөрүү):")
print(pc.pivot(index="city", columns="year", values="gender_gain").round(4).to_string())
print("\nCell тус бүрийн мөр:"); print(pc.pivot(index="city", columns="year", values="n").to_string())
print("\nХот тус бүрийн дундаж:"); print(pc.groupby("city")[["acc", "gender_gain"]].mean().round(4).to_string())
print("Хамгийн сул хот:", pc.groupby("city")["acc"].mean().idxmin())

Accuracy (хот x жил):
year           2024    2025    2026
city                               
Дархан       0.8600  0.8500  0.8550
Улаанбаатар  0.8967  0.8900  0.8967
Ховд         0.6667  0.6556  0.7444
Эрдэнэт      0.8400  0.7800  0.8350

Accuracy gender-тэй:
year           2024   2025    2026
city                              
Дархан       0.8600  0.845  0.8600
Улаанбаатар  0.8967  0.890  0.9000
Ховд         0.6667  0.600  0.6444
Эрдэнэт      0.8750  0.850  0.9100

gender_gain (accuracy-ийн зөрүү):
year          2024    2025    2026
city                              
Дархан      -0.000 -0.0050  0.0050
Улаанбаатар  0.000  0.0000  0.0033
Ховд        -0.000 -0.0556 -0.1000
Эрдэнэт      0.035  0.0700  0.0750

Cell тус бүрийн мөр:
year         2024  2025  2026
city                         
Дархан        200   200   200
Улаанбаатар   300   300   300
Ховд           90    90    90
Эрдэнэт       200   200   200

Хот тус бүрийн дундаж:
                acc  gender_gain
city                      

## Тайлан бичих заавар

Тайланг **`Lab9_Тайлан_загвар.docx`** (Лабораторийн тайлангийн загвар №9) файл дээр бичнэ. Загварын 5 хэсэг дээр энэ ноутбукийн агуулга дараах байдлаар тусна.

| Тайлангийн хэсэг | Энэ лабораторид юу бичих вэ? |
|---|---|
| 1. Зорилго | Лабын зорилго (cell, cuboid, hierarchy, measure; cube-ийн тооцоолол; cube space дахь mining). Хүрсэн эсэхээ 5-р хэсэгт бичнэ |
| 2. Онол | Лекцийн слайдын ойлголтуудыг өөрийн үгээр товч (cell, cuboid, ∏(Lᵢ + 1), concept hierarchy, measure-ийн 3 төрөл, materialization, Iceberg, BUC, closed cell, ROLAP/MOLAP, SelfExp/InExp, prediction cube) |
| 3. Аргачлал | Ноутбукийн **чухал 3-5 код** (жишээ нь `cube_groupby`, `buc`, `is_closed`, SelfExp/InExp, `cell_acc`) ба тус бүр ямар онолыг хэрхэн хэрэгжүүлснийг 2-3 өгүүлбэрээр |
| 4. Үр дүн | TASK 1-13-ын үр дүнгийн хүснэгт (загварт бэлэн), тохиромжтой график, хүснэгтийн зураг |
| 5. Дүгнэлт | Юу ойлгосон, юу хэцүү байсан, яаж шийдсэн. Доорх асуултуудад хариулна |
| Хавсралт | A. `Lab9_CubeSpace.ipynb`, `cs_*.csv`; B. git log; C. Python хувилбар, сангууд; D. GitHub холбоос |

### Дүгнэлт хэсгийн асуултууд
1. Аль даалгавар хамгийн хэцүү байсан бэ? Яаж шийдсэн бэ?
2. Antimonotonicity-г өөрийн үгээр тайлбарла. BUC-д pruning яагаад зөв байдаг вэ?
3. Cube-ийг бүгдийг нь тооцохын оронд Iceberg, closed cell, partial materialization ашиглах нь яагаад хэрэгтэй вэ? (3-5 өгүүлбэр)
4. Өөрийн салбарт (борлуулалт, сургалт, эрүүл мэнд гэх мэт) discovery-driven exploration-г хэрхэн хэрэглэх вэ?